In [2]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

GPU available: True
Device: Tesla T4


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Check the folder structure
for root, dirs, files in os.walk("/content/drive/MyDrive/movie-recommender/"):
    print(root)
    for f in files:
        print(f"  └── {f}")

In [9]:
# Force Drive to sync
import subprocess
subprocess.run(["ls", "-la", "/content/drive/MyDrive/movie-recommender/"])

import shutil, os

# Clear the stale mount point
shutil.rmtree("/content/drive", ignore_errors=True)
os.makedirs("/content/drive", exist_ok=True)

# Now mount fresh
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [11]:
import os
for item in os.listdir("/content/drive/MyDrive/movie-recommender/"):
    print(item)

models


In [12]:
from google.colab import files
import os

os.makedirs("/content/data/processed", exist_ok=True)
os.makedirs("/content/models", exist_ok=True)

print("📂 Select train.parquet")
uploaded = files.upload()

📂 Select train.parquet


Saving train.parquet to train.parquet


In [14]:
print("📂 Select test.parquet")
uploaded = files.upload()

📂 Select test.parquet


Saving test.parquet to test.parquet


In [15]:
import shutil

for f in ["train.parquet", "test.parquet"]:
    if os.path.exists(f"/content/{f}"):
        shutil.move(f"/content/{f}", f"/content/data/processed/{f}")

print(os.listdir("/content/data/processed"))
# Should show: ['train.parquet', 'test.parquet']

['train.parquet', 'test.parquet']


In [16]:
BASE      = "/content"
PROC_DIR  = f"{BASE}/data/processed/"
MODEL_DIR = f"{BASE}/models/"

print("✅ Paths set")
print(os.listdir(PROC_DIR))

✅ Paths set
['train.parquet', 'test.parquet']


In [17]:
import os

BASE      = "/content"
DATA_DIR  = f"{BASE}/data/raw/"
PROC_DIR  = f"{BASE}/data/processed/"
MODEL_DIR = f"{BASE}/models/"

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(PROC_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("✅ Paths set to local Colab storage")

✅ Paths set to local Colab storage


In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm.notebook import tqdm
import joblib, warnings

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using: {device}")

✅ Using: cuda


In [19]:
#Load Processed data

train = pd.read_parquet(PROC_DIR + "train.parquet")
test  = pd.read_parquet(PROC_DIR + "test.parquet")

print(f"Train: {len(train):,} | Test: {len(test):,}")

Train: 19,784,257 | Test: 5,026,226


In [20]:
#encode IDs

user2idx  = {u: i for i, u in enumerate(train["userId"].unique())}
movie2idx = {m: i for i, m in enumerate(train["movieId"].unique())}

train["user_idx"]  = train["userId"].map(user2idx)
train["movie_idx"] = train["movieId"].map(movie2idx)

test["user_idx"]   = test["userId"].map(user2idx)
test["movie_idx"]  = test["movieId"].map(movie2idx)
test_enc = test.dropna(subset=["user_idx","movie_idx"]).copy()
test_enc[["user_idx","movie_idx"]] = test_enc[["user_idx","movie_idx"]].astype(int)

N_USERS, N_MOVIES = len(user2idx), len(movie2idx)
print(f"Users: {N_USERS:,} | Movies: {N_MOVIES:,}")
print(f"Test rows after filter: {len(test_enc):,}")

Users: 162,541 | Movies: 18,429
Test rows after filter: 5,026,171


In [23]:
#pytorch Dataset

EMBED_DIM  = 64
HIDDEN     = [128, 64, 32]
DROPOUT    = 0.2
LR         = 1e-3
BATCH_SIZE = 1024   # larger batch = faster on GPU
EPOCHS     = 3

class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users   = torch.LongTensor(df["user_idx"].values)
        self.movies  = torch.LongTensor(df["movie_idx"].values)
        self.ratings = torch.FloatTensor(df["rating"].values)
    def __len__(self): return len(self.ratings)
    def __getitem__(self, i): return self.users[i], self.movies[i], self.ratings[i]

train_dl = DataLoader(RatingsDataset(train),    batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_dl  = DataLoader(RatingsDataset(test_enc), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_dl)} | Test batches: {len(test_dl)}")

Train batches: 19321 | Test batches: 4909


In [24]:
#NCF Model

class NCFNet(nn.Module):
    def __init__(self, n_users, n_items, embed_dim, hidden, dropout):
        super().__init__()
        self.gmf_u = nn.Embedding(n_users, embed_dim)
        self.gmf_i = nn.Embedding(n_items, embed_dim)
        self.mlp_u = nn.Embedding(n_users, embed_dim)
        self.mlp_i = nn.Embedding(n_items, embed_dim)
        layers, d = [], embed_dim * 2
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.mlp = nn.Sequential(*layers)
        self.out  = nn.Linear(embed_dim + hidden[-1], 1)
        for emb in [self.gmf_u, self.gmf_i, self.mlp_u, self.mlp_i]:
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, u, i):
        gmf = self.gmf_u(u) * self.gmf_i(i)
        mlp = self.mlp(torch.cat([self.mlp_u(u), self.mlp_i(i)], dim=1))
        return torch.sigmoid(self.out(torch.cat([gmf, mlp], dim=1)).squeeze()) * 4.5 + 0.5

model     = NCFNet(N_USERS, N_MOVIES, EMBED_DIM, HIDDEN, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
print(f"✅ NCF ready | Params: {sum(p.numel() for p in model.parameters()):,}")

✅ NCF ready | Params: 23,191,105


In [25]:
# Train NCF

train_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0
    for u, i, r in tqdm(train_dl, desc=f"Epoch {epoch}/{EPOCHS}"):
        u, i, r = u.to(device), i.to(device), r.to(device)
        optimizer.zero_grad()
        loss = criterion(model(u, i), r)
        loss.backward()
        optimizer.step()
        total += loss.item()
    avg = total / len(train_dl)
    train_losses.append(avg)
    print(f"Epoch {epoch}: Loss={avg:.4f} | RMSE≈{avg**0.5:.4f}")

print("✅ NCF training complete")

Epoch 1/3:   0%|          | 0/19321 [00:00<?, ?it/s]

Epoch 1: Loss=0.6946 | RMSE≈0.8334


Epoch 2/3:   0%|          | 0/19321 [00:00<?, ?it/s]

Epoch 2: Loss=0.5262 | RMSE≈0.7254


Epoch 3/3:   0%|          | 0/19321 [00:00<?, ?it/s]

Epoch 3: Loss=0.4464 | RMSE≈0.6682
✅ NCF training complete


In [26]:
#Evaluate NCF model

model.eval()
preds_ncf = []
with torch.no_grad():
    for u, i, _ in test_dl:
        preds_ncf.extend(model(u.to(device), i.to(device)).cpu().numpy().tolist())

preds_ncf = np.clip(preds_ncf, 0.5, 5.0)
y_true    = test_enc["rating"].values

rmse_ncf = np.sqrt(mean_squared_error(y_true, preds_ncf))
mae_ncf  = mean_absolute_error(y_true, preds_ncf)

print("=" * 40)
print(f"NCF → RMSE: {rmse_ncf:.4f} | MAE: {mae_ncf:.4f}")
print("=" * 40)

ncf_results = {"model": "NCF", "RMSE": rmse_ncf, "MAE": mae_ncf}

NCF → RMSE: 0.8245 | MAE: 0.6202


In [27]:
#save NCF to drive

# Save model weights
torch.save(model.state_dict(), MODEL_DIR + "ncf.pt")

# Save predictions and metadata
joblib.dump({
    "user2idx"  : user2idx,
    "movie2idx" : movie2idx,
    "preds"     : preds_ncf,
    "results"   : ncf_results,
    "test_index": test_enc.index.tolist()
}, MODEL_DIR + "ncf_meta.joblib")

print("✅ NCF saved to Google Drive")

✅ NCF saved to Google Drive


In [29]:
#ALS model
!pip install implicit -q
print("✅ implicit installed")
from scipy.sparse import csr_matrix
import implicit

ALPHA = 40.0

# Reload clean train (without user_idx/movie_idx columns)
train_raw = pd.read_parquet(PROC_DIR + "train.parquet")
test_raw  = pd.read_parquet(PROC_DIR + "test.parquet")

als_user2idx  = {u: i for i, u in enumerate(train_raw["userId"].unique())}
als_movie2idx = {m: i for i, m in enumerate(train_raw["movieId"].unique())}
als_idx2movie = {v: k for k, v in als_movie2idx.items()}

item_user_matrix = csr_matrix(
    (ALPHA * train_raw["rating"].values,
     (train_raw["movieId"].map(als_movie2idx),
      train_raw["userId"].map(als_user2idx))),
    shape=(len(als_movie2idx), len(als_user2idx))
)

als = implicit.als.AlternatingLeastSquares(
    factors=50, iterations=5, regularization=0.01, use_gpu=False
)
als.fit(item_user_matrix)
print("✅ ALS trained")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ implicit installed


  0%|          | 0/5 [00:00<?, ?it/s]

✅ ALS trained


In [31]:
#Evaluate ALS

N_USERS_ALS  = als.user_factors.shape[0]
N_MOVIES_ALS = als.item_factors.shape[0]

def predict_als(user_id, movie_id):
    u, m = als_user2idx.get(user_id), als_movie2idx.get(movie_id)
    if u is None or m is None: return 3.5
    if u >= N_USERS_ALS or m >= N_MOVIES_ALS: return 3.5   # bounds check
    score = np.dot(als.user_factors[u], als.item_factors[m])
    return float(np.clip(score, 0, 1) * 4.5 + 0.5)

test_als = test_raw[
    test_raw["userId"].isin(als_user2idx) &
    test_raw["movieId"].isin(als_movie2idx)
].sample(frac=0.2, random_state=42).copy()

print(f"Evaluating on {len(test_als):,} rows")

preds_als  = np.array([predict_als(r.userId, r.movieId) for r in tqdm(test_als.itertuples(), total=len(test_als))])
y_true_als = test_als["rating"].values

rmse_als = np.sqrt(mean_squared_error(y_true_als, preds_als))
mae_als  = mean_absolute_error(y_true_als, preds_als)

print(f"ALS → RMSE: {rmse_als:.4f} | MAE: {mae_als:.4f}")
als_results = {"model": "ALS", "RMSE": rmse_als, "MAE": mae_als}

Evaluating on 1,005,234 rows


  0%|          | 0/1005234 [00:00<?, ?it/s]

ALS → RMSE: 1.3956 | MAE: 1.0357


In [32]:
#save the model to drive

joblib.dump({
    "model"           : als,
    "user2idx"        : als_user2idx,
    "movie2idx"       : als_movie2idx,
    "idx2movie"       : als_idx2movie,
    "item_user_matrix": item_user_matrix,
    "preds"           : preds_als,
    "results"         : als_results,
    "test_index"      : test_als.index.tolist()
}, MODEL_DIR + "als.joblib")

print("✅ ALS saved to Google Drive")
print(f"\nFiles in models/: {os.listdir(MODEL_DIR)}")

✅ ALS saved to Google Drive

Files in models/: ['als.joblib', 'ncf.pt', 'ncf_meta.joblib']


In [35]:
from google.colab import files

files.download(MODEL_DIR + "als.joblib")
files.download(MODEL_DIR + "ncf_meta.joblib")
files.download(MODEL_DIR + "ncf.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>